# US FDA: AI-enabled devices and change-control plans: worked solution

Worked solution for the exercise in the lesson **[US FDA: AI-enabled devices and change-control plans](https://ramkumarmd.com/medical-ai/regulation-ethics/us-fda-ai-devices)** (*Regulation, ethics & governance* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](02-us-fda-ai-devices.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** In the United States, most AI-enabled medical devices reach the market through the FDA's 510(k) pathway by showing substantial equivalence to an existing device; novel lower-risk devices use De Novo, and the highest-risk ones need premarket approval. Some clinical decision support software is excluded from device regulation if clinicians can independently review the basis of its recommendations. Predetermined change control plans let manufacturers update AI devices within limits approved in advance. As of 2026, the FDA has authorised more than 1,600 AI-enabled devices and is developing its approach to generative AI.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Five FICTIONAL decision-support tools, and a fictional retinopathy model, for the CDS and PCCP steps.
cds_tools = [
    {"name": "StatinGuide", "description": "Shows the clinician the guideline recommendation for statin therapy, with the "
     "patient's risk factors and the guideline passage it used, for the clinician to review."},
    {"name": "ECG-Rhythm", "description": "Analyses the ECG waveform from a recorder and labels atrial fibrillation."},
    {"name": "A1cTrend", "description": "Displays a patient's HbA1c results over time with the clinic's target line."},
    {"name": "SepsisNow", "description": "Continuously scores ward patients' vital signs and alerts nurses to start the sepsis "
     "protocol within 1 hour; the score's inputs and weights are not shown."},
    {"name": "PatientDose", "description": "Tells patients at home how much to change their basal insulin from their "
     "fasting glucose readings."},
]

retinopathy_model = ("RetinaNext (fictional): a cleared AI system that grades retinal photographs for referable "
                     "diabetic retinopathy. The manufacturer wants to retrain it every year on new images from its "
                     "partner clinics, including from new camera models.")

# The FDA's AI-enabled device list: download the CSV in a web browser from the FDA's
# "Artificial Intelligence-Enabled Medical Devices" page (the site may block scripted downloads),
# then upload it here (in Colab: the folder icon on the left) as fda_ai_devices.csv.
FDA_LIST_FILE = "fda_ai_devices.csv"
print(f"{len(cds_tools)} fictional CDS tools; expected FDA list file: {FDA_LIST_FILE}")

## Exercise

**Goal:** Read FDA device records critically and apply the CDS criteria and PCCP concepts.

**Data:** The FDA's public AI-enabled device list (download) and fictional product descriptions (US government public information; course materials)

### Step 1

Download the FDA AI-enabled device list and count devices by review panel and pathway; how many are in radiology?

In [ ]:
import pandas as pd
from pathlib import Path

def load_fda_list(path=FDA_LIST_FILE):
    """Reads the FDA AI-enabled device list CSV (columns as published: Date of Final Decision, Submission Number,
    Device, Company, Panel (Lead), Primary Product Code). Header rows above the table are skipped."""
    raw = Path(path).read_text(encoding="utf-8-sig", errors="replace").splitlines()
    start = next(i for i, line in enumerate(raw) if line.startswith("Date of Final Decision"))
    from io import StringIO
    return pd.read_csv(StringIO("\n".join(raw[start:])))

def pathway(submission):
    s = str(submission).upper()
    return "De Novo" if s.startswith("DEN") else "PMA" if s.startswith("P") else "510(k)" if s.startswith("K") else "other"

if Path(FDA_LIST_FILE).exists():
    devices = load_fda_list()
    devices["pathway"] = devices["Submission Number"].map(pathway)
    print(f"{len(devices)} devices in the list")
    print(devices["pathway"].value_counts().to_string())
    print(devices["Panel (Lead)"].value_counts().head(8).to_string())
    print(f"Radiology: {(devices['Panel (Lead)'].str.contains('Radiology', case=False)).sum()} devices")
else:
    print(f"{FDA_LIST_FILE} not found: download the CSV from the FDA page in a browser and upload it, then re-run.")

We could not download the list from a script when writing this solution (the FDA site blocked automated requests),
so the code was tested on a small file with the published column names, not on the real list. The submission
number tells you the pathway: K numbers are 510(k) clearances, DEN numbers De Novo, and P numbers PMA. As the lesson
says, the large majority are 510(k) and most are in radiology; your counts will depend on the list's date.

### Step 2

Open the decision summaries of two diabetes-related devices and note what evidence supported clearance.

In [ ]:
if Path(FDA_LIST_FILE).exists():
    diabetes = devices[devices["Device"].str.contains("retin|diabet|insulin|glucose", case=False, na=False)]
    print(diabetes[["Date of Final Decision", "Submission Number", "Device", "Company", "pathway"]].to_string(index=False))

Open each device's FDA database entry from its submission number and read the decision summary. What to note:
the pathway and predicate (for 510(k)), the intended use statement, the study design (retrospective images or a
prospective study in the intended setting), the population and sites, the reference standard, the primary results
with confidence intervals, and any limitations in the labelling.

A useful comparison point is the first autonomous diabetic retinopathy system, authorised through De Novo in 2018
(DEN180001): its pivotal study enrolled 900 people with diabetes at 10 primary-care sites and reported sensitivity
87.2% and specificity 90.7% against a reading-centre reference standard (Abràmoff et al., npj Digital Medicine
2018). Many 510(k) clearances rest on smaller, retrospective comparisons with a predicate.

### Step 3

Apply the four CDS criteria to five fictional decision-support tools and explain each conclusion.

In [ ]:
CRITERIA = [
    "1. Does NOT analyse a medical image, an IVD signal, or a signal/pattern from a signal-acquisition system",
    "2. Displays, analyses or prints medical information (for example patient data, guidelines)",
    "3. Supports or provides recommendations to a health care professional",
    "4. The professional can independently review the basis of the recommendation (not rely primarily on it)",
]
judgements = {   # one True/False per criterion, with the deciding reason
    "StatinGuide": ([True, True, True, True], "meets all four: likely non-device CDS"),
    "ECG-Rhythm": ([False, False, True, False], "analyses a physiological signal: a device"),
    "A1cTrend": ([True, True, True, True], "displays data for a professional; likely non-device (it may not even be CDS)"),
    "SepsisNow": ([True, True, True, False], "time-critical alert with a hidden basis: the clinician cannot independently review it, so a device"),
    "PatientDose": ([True, True, False, False], "recommendations go to patients, not professionals: a device"),
}
for name, (met, reason) in judgements.items():
    print(f"{name:<12} {''.join('Y' if m else 'n' for m in met)}  {reason}")

All four criteria must be met for software to fall outside the device definition under the 21st Century Cures Act.
The FDA's guidance (revised January 2026, per the lesson) explains how it reads them; the conclusions above are
teaching judgements on fictional products. Note how often criterion 4 decides the case: urgency and opacity both
make independent review unrealistic.

### Step 4

Write a short PCCP outline for a fictional retinopathy model that will be retrained yearly on new images.

PCCP outline for RetinaNext (fictional), yearly retraining:

**Description of modifications**
- Annual retraining of the grading model on new de-identified images from partner clinics, including images from
  additional camera models listed in an appendix.
- No change to the intended use, the user, the output categories or the operating threshold's target performance.

**Modification protocol**
- Data: inclusion rules for new images; reference standard grading by certified graders; a locked, sequestered
  test set (plus a new test set for each added camera) never used for training.
- Training: the same architecture family and training procedure; documented random seeds and versions.
- Acceptance criteria: sensitivity and specificity for referable retinopathy no lower than the cleared version's
  lower confidence bounds, overall and in each subgroup (camera model, site, age, sex, ethnicity where recorded),
  and image-gradability rate within a set margin.
- Implementation: staged release, the ability to roll back to the previous version, and user notification of each
  update with a summary of performance.

**Impact assessment**
- Benefits: better performance on new cameras and populations. Risks: performance falling in a subgroup, drift in
  gradability, users not noticing a change. Controls: subgroup acceptance criteria, post-release monitoring of
  referral rates and graded samples, and a stop rule.

Changes outside this plan (a new output, autonomous use in a new setting, a different reference standard) would
need a new submission.

---

Lesson: [https://ramkumarmd.com/medical-ai/regulation-ethics/us-fda-ai-devices](https://ramkumarmd.com/medical-ai/regulation-ethics/us-fda-ai-devices). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.